# 02 · Parsing y chunking: conservar evidencia antes de crear vectores

**Objetivo:** construir chunks reales, reproducibles y trazables; estudiar cómo
tamaño, overlap y estructura afectan a fragmentación, duplicación y límites del
encoder. No evaluaremos retrieval ni respuestas médicas todavía.

Corpus predeterminado: **PI-RADS v2.1 (2019), ACR–ESUR–AdMeTech**, manual de
76 páginas aportado por el usuario con cobertura de uso confirmada. ESUR-2012 sigue
disponible como comparación histórica: seleccionar su ID en la configuración.

**Abrir en VSCode:** seleccionar `.venv/Scripts/python.exe` como kernel y ejecutar
de arriba abajo. No es necesario abrir JupyterLab. Este notebook no depende de
haber ejecutado 00 o 01: lee el original y su manifiesto.

Preparación única desde la raíz: `uv sync --locked` y
`uv run python scripts/prepare_tokenizer.py`. Sólo descarga el tokenizer fijado,
no pesos de XLM-RoBERTa. La ejecución predeterminada es local y no usa `.env`,
Ollama, un LLM ni embeddings. El apartado semántico es opcional y queda desactivado.

## 1. Fundamento y preguntas experimentales

Un chunk es una unidad de recuperación, no necesariamente una página ni un párrafo.
Si una condición queda en un chunk y su nota en otro, recuperar sólo uno puede
resultar insuficiente aunque ambos contengan palabras relevantes.

- **Tamaño fijo:** controla el presupuesto y puede cortar relaciones.
- **Overlap:** duplica contenido para amortiguar cortes, a cambio de almacenamiento
  y contexto redundante. No garantiza conservar una tabla entera.
- **Estructura:** respeta párrafos/secciones y vincula tablas multipágina, pero
  puede producir unidades mayores que la ventana del encoder.
- **Semántico:** propone límites mediante representaciones de contenido. Necesita
  un encoder; no basta con tokenizar. El coste y el umbral requieren evaluación.

No confundamos **tokenizer** con **encoder**. El primero convierte texto en IDs y
offsets; el segundo calcula vectores usando pesos entrenados. Aquí usaremos el
tokenizer real de E5 para el presupuesto del futuro encoder, cuya ventana es 512.

$$T(\texttt{passage: } + c) \le B$$

`T` incluye tokens especiales y no trunca. Compararemos B = 256, 512 y 768. La
configuración 768 es un control que excede la ventana E5: sus chunks se marcan,
no se presentan como entradas utilizables completas por ese encoder.

In [ ]:
import hashlib
import json
import math
import platform
import re
import time
from datetime import UTC, datetime
from importlib.metadata import version

import matplotlib.pyplot as plt
import pandas as pd
import pdfplumber
import pypdfium2 as pdfium
from IPython.display import Markdown, display
from pypdf import PdfReader

from radiology_rag.evaluation.parsing import extraction_scores, evidence_preservation
from radiology_rag.ingestion.catalog import load_catalog, sha256_file
from radiology_rag.ingestion.chunking import (
    TokenCounter, chunk_metrics, fixed_chunks, flatten_blocks, make_chunk, structural_chunks,
)
from radiology_rag.ingestion.parsing import fingerprint, parse_regions
from radiology_rag.paths import project_root, within

ROOT = project_root()
RUN_SEMANTIC = False
ALLOW_ENCODER_DOWNLOAD = False  # Únicamente se consulta en el apartado opcional.
catalog = load_catalog(ROOT / "data/catalog/documents.v1.json")
DOCUMENT_ID = catalog.get("active_document_id", "esur-2012")  # O fijar "esur-2012".
doc = next(d for d in catalog["documents"] if d["id"] == DOCUMENT_ID)
PDF_PATH = within(ROOT, doc["local_path"])
manifest = json.loads(PDF_PATH.with_suffix(".source.json").read_text(encoding="utf-8"))
assert sha256_file(PDF_PATH) == manifest["sha256"] == doc["expected_sha256"]
processing = doc["processing"]
layout_path = within(ROOT, processing["layout_path"])
reference_path = within(ROOT, processing["reference_path"])
layout = json.loads(layout_path.read_text(encoding="utf-8"))
reference = json.loads(reference_path.read_text(encoding="utf-8"))
assert reference["source_sha256"] == manifest["sha256"]
counter = TokenCounter.from_project(ROOT)
OUT = ROOT / "data/processed" / DOCUMENT_ID / "02_chunking"
OUT.mkdir(parents=True, exist_ok=True)
environment = {"python": platform.python_version(), **{
    p: version(p) for p in ("tokenizers", "pypdf", "pdfplumber", "pandas", "matplotlib")}}
display(pd.Series({"document": DOCUMENT_ID, "document_version": doc["version"],
                   "tokenizer": counter.spec["repository"],
                   "revision": counter.spec["revision"],
                   "encoder_window": counter.spec["encoder_max_tokens"],
                   "encoder_executed": False}))

## 2. Referencias de extracción independientes de los parsers

Cada documento tiene **10 microregiones** transcritas del render, con coordenadas,
atribución y hash en el archivo `reference_path` seleccionado desde el catálogo.
Incluyen encabezados, tablas y un control de contenido dentro de una imagen.
Son referencias revisadas visualmente por el agente, **pendientes de revisión humana**;
no constituyen un gold standard validado ni preguntas clínicas.

La muestra es pequeña y seleccionada para estudiar dificultades conocidas. CER y WER
cuentan ediciones sobre caracteres/palabras normalizados; pueden exceder 1 si el crop
incluye texto extra. Un 0 no valida relaciones entre filas, condiciones o notas.

In [ ]:
# Ver las microregiones originales, incluyendo la imagen que no se recupera como texto.
figure, axes = plt.subplots(5, 2, figsize=(16, 9))
pdf_render = pdfium.PdfDocument(PDF_PATH)
for ax, region in zip(axes.flat, reference["regions"], strict=True):
    page = pdf_render[region["page_number"] - 1]
    bitmap = page.render(scale=3)
    image = bitmap.to_pil().copy()
    bitmap.close()
    page.close()
    box = tuple(round(value * 3) for value in region["bbox"])
    ax.imshow(image.crop(box))
    ax.set_title(f"p{region['page_number']} · {region['id']}\n{region['kind']}", fontsize=10)
    ax.axis("off")
pdf_render.close()
figure.tight_layout()
figure.savefig(OUT / "reference_regions.png", dpi=150)
plt.show()

In [ ]:
comparison_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for region in reference["regions"]:
        # within_bbox excluye caracteres parcialmente fuera del crop de referencia.
        crop = pdf.pages[region["page_number"] - 1].within_bbox(tuple(region["bbox"]))
        for name, options in [("default", {}), ("x_tolerance=1", {"x_tolerance": 1})]:
            actual = crop.extract_text(**options) or ""
            comparison_rows.append({
                "region_id": region["id"], "page": region["page_number"],
                "kind": region["kind"], "parser_config": name,
                **extraction_scores(region["expected_text"], actual),
                "expected": region["expected_text"], "observed": actual,
            })
parsing_comparison = pd.DataFrame(comparison_rows)
display(parsing_comparison[["region_id", "parser_config", "exact_normalized",
                            "character_error_rate", "word_error_rate"]].round(3))
display(parsing_comparison[parsing_comparison.kind == "image_text"][[
    "expected", "observed", "parser_config"]])
parsing_comparison.to_csv(OUT / "parsing_reference_comparison.csv", index=False)

**Interpretación:** contrasta cada crop con `expected` y `observed`. Si la referencia
o su bounding box es incorrecta, corregirla a partir de la imagen y versionarla;
no ajustar la referencia a lo que devuelve un parser. Un texto ausente dentro de
una imagen no se arregla cambiando el tamaño del chunk.

## 3. Parsing por regiones y estructura: una política explícita

El mapa `layout_path` declara regiones y su orden, secciones y grupos de tabla.
PI-RADS usa una columna y tablas; ESUR combina columnas y tablas a todo el ancho. Está ligado al hash de este archivo; otro documento exige otro mapa.
Las secciones están revisadas por el agente y pendientes de revisión humana.

La extracción usa `x_tolerance=1`, conserva saltos de línea y no corrige términos
médicos ni une automáticamente palabras con guion. Los párrafos se proponen con
huecos verticales y títulos en negrita. Las tablas permanecen como bloques de texto
vinculados: **todavía no son tablas semánticas validadas**.

In [ ]:
# Implementación geométrica de una región: no llamamos a un framework de parsing.
sample_page, sample_region = next((p["page_number"], r) for p in layout["pages"]
    for r in p["regions"] if r["id"] == processing["sample_region_id"])
with pdfplumber.open(PDF_PATH) as pdf:
    crop = pdf.pages[sample_page - 1].crop(tuple(sample_region["bbox"]))
    lines = crop.extract_text_lines(x_tolerance=1, return_chars=True)
simple_paragraphs, current = [], []
for line in lines:
    if current and line["top"] - current[-1]["bottom"] > 5:
        simple_paragraphs.append("\n".join(item["text"] for item in current))
        current = []
    current.append(line)
if current:
    simple_paragraphs.append("\n".join(item["text"] for item in current))
print("Párrafos propuestos en la región:", len(simple_paragraphs))
print(simple_paragraphs[0])

In [ ]:
# La versión reutilizable añade la regla de negrita y toda la procedencia.
started = time.perf_counter()
blocks, unresolved_images, page_audit = parse_regions(PDF_PATH, doc, layout)
parsing_seconds = time.perf_counter() - started
corpus_text, located = flatten_blocks(blocks)
display(pd.DataFrame(blocks).groupby(["page_number", "kind"]).size().unstack(fill_value=0))
print("Caracteres del corpus seleccionado:", len(corpus_text))
print("Objetos visuales sin transcripción estructurada:", len(unresolved_images))
display(pd.DataFrame([{
    "page": row["page_number"], "all_words": row["total_words"],
    "words_outside_map": len(row["words_outside_regions"]),
} for row in page_audit]))

**Control de pérdidas:** el audit conserva palabras fuera del mapa para revisar
cabeceras, pies y posibles omisiones. Referencias, avisos, ejemplos cumplimentados y
pies de figura se conservan como bloques, pero se excluyen del corpus de recuperación.
Las imágenes informativas quedan en un manifiesto de pendientes, sin inventar
transcripciones. Las cabeceras y portadas decorativas se registran aparte en el audit.

La cobertura posterior se mide **respecto al texto seleccionado**, no al contenido
completo del PDF. Puede ser 100% aunque falten imágenes por OCR. En PI-RADS la tabla
del léxico se conserva por página como padre; todavía hay que validar cada relación.
La capitular de introducción se reúne mediante una regla explícita del mapa.

## 4. Usar ya el tokenizer real: IDs, offsets y presupuesto

La revisión fijada de `intfloat/multilingual-e5-base` está en el catálogo de tokenizers.
El archivo contiene reglas/vocabulario y postprocesado, no el encoder entrenado.
Desactivamos padding/truncación para medir sin perder entradas. La ventana de E5
incluye el prefijo `passage:` y tokens especiales.

Inspecciona cómo una palabra, puntuación o símbolo puede ocupar varios tokens. No
existe una conversión constante de caracteres o palabras a tokens.

In [ ]:
example = "Español, ADC ≥ b800 y diffusion-weighted MRI."
encoded = counter.tokenizer.encode(example, add_special_tokens=False)
display(pd.DataFrame({"id": encoded.ids, "token": encoded.tokens,
                      "offset": encoded.offsets,
                      "original_slice": [example[a:b] for a, b in encoded.offsets]}))
print("Tokens sin prefijo/especiales:", len(encoded.ids))
print("Tokens con passage y especiales:", counter.count(example))
print("Overhead de prefijo y especiales:", counter.count(""))
print("Modelo encoder ejecutado: NO")

## 5. Cortes fijos desde cero

Trabajamos sobre un corpus común con offsets de caracteres. El corte se propone con
offsets del tokenizer y se vuelve a contar el substring con prefijo/especiales.
No reconstruimos el texto con `decode`: podría normalizar espacios o caracteres y
romper la relación con la fuente. El overlap se expresa como objetivo de tokens;
el coste real de duplicación se mide sobre los caracteres originales.

In [ ]:
def visible_token_windows(text, budget, overlap=0):
    start = 0
    overhead = counter.count("")
    while start < len(text):
        offsets = counter.tokenizer.encode(text[start:], add_special_tokens=False).offsets
        if not offsets:
            break
        limit = min(len(offsets), budget - overhead)
        end = len(text) if limit == len(offsets) else start + offsets[limit][0]
        while end > start and counter.count(text[start:end]) > budget:
            limit -= 1
            end = start + offsets[limit][0]
        if end <= start:
            raise ValueError("Presupuesto insuficiente para avanzar sin perder texto")
        yield start, end, counter.count(text[start:end])
        if end == len(text):
            break
        current = counter.tokenizer.encode(text[start:end], add_special_tokens=False).offsets
        keep = min(len(current) - 1, math.floor(budget * overlap))
        next_start = start + current[-keep][0] if keep else end
        start = next_start if next_start > start else end

intervals = list(visible_token_windows(corpus_text, 256, 0.15))
assert intervals == list(counter.windows(corpus_text, 256, 0.15))
display(pd.DataFrame(intervals[:5], columns=["start", "end", "total_tokens"]))
a, b, _ = intervals[0]
print(corpus_text[a:b][-600:])

In [ ]:
experiments, timings = {}, {}
for budget in (256, 512, 768):
    for overlap in (0, 0.15):
        name = f"fixed_{budget}_{int(overlap * 100)}"
        started = time.perf_counter()
        experiments[name] = [
            make_chunk(corpus_text, located, counter, a, b, "fixed_tokens",
                       {"budget": budget, "overlap": overlap})
            for a, b, _ in visible_token_windows(corpus_text, budget, overlap)
        ]
        timings[name] = time.perf_counter() - started
        # Contrastar versión educativa y componente reutilizable sobre el corpus real.
        assert [c["chunk_id"] for c in experiments[name]] == [c["chunk_id"] for c in
            fixed_chunks(corpus_text, located, counter, budget=budget, overlap=overlap)]
print("Configuraciones fijas ejecutadas:", list(experiments))

## 6. Chunking estructural: párrafos, secciones y tablas vinculadas

Primero construimos unidades indivisibles: párrafos y tablas. Un `group_id`
vincula partes de una misma tabla; por ejemplo Table 3 de ESUR entre páginas 8–9.
En PI-RADS se mantienen juntos los cuadros de evaluación y cada página del léxico. Se agrupan
párrafos consecutivos de la misma sección mientras caben. Las tablas quedan aparte.

Si una tabla supera el presupuesto, conservamos su bloque padre **sin truncar** y
lo señalamos como demasiado largo cuando exceda 512. Eso protege el original, pero
todavía exige diseñar hijos por filas/condiciones para recuperarlo con E5.

In [ ]:
# Algoritmo completo, visible, aplicado al mismo corpus que los cortes fijos.
def visible_structural_intervals(budget):
    atoms = []
    for block in located:
        if (atoms and block["kind"] == "table" and block["group_id"]
                and atoms[-1]["group_id"] == block["group_id"]):
            atoms[-1]["end"] = block["end"]
        else:
            atoms.append({k: block[k] for k in
                          ("start", "end", "kind", "group_id", "section_path")})
    output, pending = [], None
    for atom in atoms:
        start, end = atom["start"], atom["end"]
        if atom["kind"] == "table":
            if pending:
                output.append((pending["start"], pending["end"]))
                pending = None
            output.append((start, end))
            continue
        same_section = pending and pending["section_path"] == atom["section_path"]
        fits = pending and counter.count(corpus_text[pending["start"]:end]) <= budget
        if pending and (not same_section or not fits):
            output.append((pending["start"], pending["end"]))
            pending = None
        if counter.count(corpus_text[start:end]) > budget:
            output.extend((start + a, start + b)
                          for a, b, _ in visible_token_windows(corpus_text[start:end], budget))
        elif pending:
            pending["end"] = end
        else:
            pending = {"start": start, "end": end, "section_path": atom["section_path"]}
    if pending:
        output.append((pending["start"], pending["end"]))
    return output

for budget in (256, 512, 768):
    started = time.perf_counter()
    chunks = structural_chunks(corpus_text, located, counter, budget=budget)
    assert [(c["start"], c["end"]) for c in chunks] == visible_structural_intervals(budget)
    name = f"structural_{budget}"
    experiments[name] = chunks
    timings[name] = time.perf_counter() - started
table_chunks = [c for c in experiments["structural_512"]
                if any(span["kind"] == "table" for span in c["spans"])]
display(pd.DataFrame([{"pages": c["pages"], "tokens": c["token_count_with_passage_prefix"],
                       "exceeds_512": c["exceeds_encoder_window"],
                       "unresolved_images": c["unresolved_objects"]} for c in table_chunks]))

## 7. Experimento opcional de límites semánticos

Esta sección sí requiere un **encoder local**. Por defecto no se ejecuta y no habrá
una fila semántica en los resultados. Si quieres activarla, instala el extra con
`uv sync --extra embeddings`, cambia `RUN_SEMANTIC=True` y permite una descarga
explícita con `ALLOW_ENCODER_DOWNLOAD=True` sólo si no dispones ya de pesos locales.
El modelo puede ocupar más de 1 GB y consumir RAM/CPU; no requiere Ollama ni API key.

Unidad: párrafos cortos. Se normalizan vectores y se calcula
$d_i = 1-\langle e_i,e_{i+1}\rangle$. Distancias sobre el percentil 80 proponen cortes
adicionales a estructura/ventana. Para similitud simétrica E5 recomienda `query:`;
retrieval de pasajes empleará `passage:` en 03. No se vectorizan párrafos que excedan
512 con su prefijo. Tablas siguen siendo atómicas, con los mismos avisos de tamaño.

El umbral 80 es una hipótesis, no un óptimo. No ajustarlo con el futuro conjunto de
evaluación. Estos vectores de párrafos no constituyen un índice persistente de RAG.

In [ ]:
semantic_status = {"executed": False, "reason": "RUN_SEMANTIC=False"}
if RUN_SEMANTIC:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as error:
        raise RuntimeError("Instalar: uv sync --extra embeddings") from error
    encoder = SentenceTransformer(
        counter.spec["repository"], revision=counter.spec["revision"], device="cpu",
        cache_folder=str(ROOT / "data/raw/models"), trust_remote_code=False,
        local_files_only=not ALLOW_ENCODER_DOWNLOAD,
    )
    assert encoder.max_seq_length == 512, "Cambió la ventana: revisar el experimento"
    prose = [b for b in located if b["kind"] == "prose" and
             len(counter.tokenizer.encode("query: " + b["text"]).ids) <= 512]
    started = time.perf_counter()
    vectors = encoder.encode(["query: " + b["text"] for b in prose],
                             normalize_embeddings=True, show_progress_bar=False)
    candidates = []
    for index in range(len(prose) - 1):
        left, right = prose[index], prose[index + 1]
        # Sólo párrafos realmente consecutivos de la misma sección.
        between = [b for b in located if left["end"] < b["start"] < right["start"]]
        if not between and left["section_path"] == right["section_path"]:
            candidates.append((1 - float(vectors[index] @ vectors[index + 1]), right["start"]))
    distances = sorted(distance for distance, _ in candidates)
    threshold = distances[max(0, math.ceil(len(distances) * 0.8) - 1)] if distances else 1
    cuts = [position for distance, position in candidates if distance > threshold]
    chunks = structural_chunks(corpus_text, located, counter, budget=512, forced_breaks=cuts)
    experiments["semantic_q80_512"] = chunks
    timings["semantic_q80_512"] = time.perf_counter() - started
    semantic_status = {"executed": True, "model": counter.spec["repository"],
                       "revision": counter.spec["revision"], "threshold": threshold,
                       "forced_breaks": cuts, "paragraphs_encoded": len(prose),
                       "note": "Sin cambios respecto a estructural si no hay cortes adicionales"}
print(semantic_status)

## 8. Medir mecánica de chunking y evidencia preservada

Medimos número y longitudes, mezcla de secciones, párrafos partidos y conservación
de grupos de tabla. Sobre caracteres no blancos del **corpus seleccionado**:

$$coverage=\frac{\#\text{caracteres cubiertos}}{\#\text{caracteres del corpus}},\quad
copy\_factor=\frac{\sum_c \#\text{caracteres de }c}{\#\text{caracteres cubiertos}}.$$

Sin overlap se espera factor 1; con overlap, mayor que 1. La cobertura no evalúa la
fidelidad del parsing. Tampoco son Recall@k, Precision@k ni calidad de respuesta:
no existe aún un recuperador o una evaluación de QA.

In [ ]:
metric_rows = [{"experiment": name, **chunk_metrics(chunks, corpus_text, located),
                "seconds": timings[name]} for name, chunks in experiments.items()]
metrics = pd.DataFrame(metric_rows)
display(metrics.round(3))
assert (metrics.coverage_of_selected_text == 1).all(), "Se perdió texto seleccionado"
assert not any(c["mixed_sections"] for c in experiments["structural_512"])
assert all(c["token_count_with_passage_prefix"] <= 512
           for c in experiments["fixed_512_15"])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
names = metrics.experiment.tolist()
for ax, column, title in zip(axes.flat,
    ["chunks", "character_copy_factor", "paragraphs_split", "chunks_over_encoder_window"],
    ["Número de chunks", "Copias por carácter cubierto", "Párrafos sin un chunk que los contenga",
     "Chunks que exceden la ventana E5"], strict=True):
    ax.bar(names, metrics[column])
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=65)
fig.tight_layout()
fig.savefig(OUT / "chunking_comparison.png", dpi=140)
plt.show()

In [ ]:
evidence_cases = reference["evidence_cases"]

evidence_rows = []
for name, chunks in experiments.items():
    evidence_rows.extend({"experiment": name, **result} for result in
                         evidence_preservation(chunks, corpus_text, evidence_cases, located))
evidence = pd.DataFrame(evidence_rows)
display(evidence.pivot(index="experiment", columns="case_id",
                       values="single_chunk_contains_evidence"))
display(evidence.pivot(index="experiment", columns="case_id",
                       values="single_chunk_within_encoder_window"))

**Cómo interpretar las dos tablas:** un padre estructural puede conservar la tabla
multipágina y, aun así, no caber en E5. La segunda tabla muestra esa limitación sin
simular una mejora utilizable. Las anclas son micropruebas de contención, no ejemplos
clínicos validados. Si falta la ancla de una imagen en el parsing, ningún chunker
textual puede recuperarla. La solución requiere otro paso de extracción/revisión.

## 9. Inspeccionar un chunk y sus citas antes de exportar

Las referencias usan página física y etiqueta editorial revisada en el mapa. Los spans indican
offsets del bloque original y bounding box **del bloque entero**: no fingimos tener
geometría exacta de cada carácter del substring. Un ID incorpora documento, versión,
hash, corpus, configuración, tokenizer, offsets y contenido.

In [ ]:
inspection_blocks = {b["block_id"] for b in located
    if b["group_id"] == processing["inspection_group_id"]}
example_chunk = next(c for c in experiments["structural_512"]
    if any(s["block_id"] in inspection_blocks for s in c["spans"]))
display(pd.Series({key: example_chunk[key] for key in (
    "chunk_id", "document_version", "pages", "token_count_with_passage_prefix",
    "exceeds_encoder_window", "section_paths", "unresolved_objects")}))
display(pd.DataFrame(example_chunk["spans"])[["page_number", "page_label", "region_id",
                                            "block_start", "block_end", "bbox_scope"]])
print(example_chunk["text"][:650])
print("Revisión humana de relaciones tabulares: PENDIENTE")

In [ ]:
# Publicar el mismo resultado que la CLI; contrastar todos los IDs calculados arriba.
if not RUN_SEMANTIC:
    from radiology_rag.ingestion.pipeline import process_document
    report = process_document(ROOT, DOCUMENT_ID)
    pointer = json.loads((OUT / "latest.json").read_text(encoding="utf-8"))
    RUN_OUT = within(ROOT, pointer["path"])
    for name, chunks in experiments.items():
        saved = [json.loads(line) for line in
                 (RUN_OUT / f"chunks_{name}.jsonl").read_text(encoding="utf-8").splitlines()]
        assert [c["chunk_id"] for c in chunks] == [c["chunk_id"] for c in saved]
    print("Ejecución completa y verificada:", RUN_OUT)
    print("Índice vectorial creado: NO")
else:
    def write_jsonl(path, records):
        path.write_text("".join(json.dumps(record, ensure_ascii=False) + "\n" for record in records),
                        encoding="utf-8")

    corpus_hash = fingerprint([(b["block_id"], b["start"], b["end"]) for b in located])
    run_identity = {"source_sha256": manifest["sha256"], "corpus_fingerprint": corpus_hash,
                    "layout_fingerprint": fingerprint(layout),
                    "references_fingerprint": fingerprint(reference),
                    "tokenizer_sha256": counter.spec["sha256"],
                    "environment": environment, "semantic": semantic_status,
                    "experiment_names": list(experiments)}
    run_identity["implementation_sha256"] = {
        relative: sha256_file(ROOT / relative) for relative in (
            "notebooks/02_parsing_and_chunking.ipynb",
            "src/radiology_rag/ingestion/parsing.py",
            "src/radiology_rag/ingestion/chunking.py",
            "src/radiology_rag/evaluation/parsing.py", "uv.lock")}
    run_id = fingerprint(run_identity)[:16]
    RUN_OUT = OUT / "semantic_runs" / run_id
    RUN_OUT.mkdir(parents=True, exist_ok=True)
    write_jsonl(RUN_OUT / "blocks.jsonl", blocks)
    write_jsonl(RUN_OUT / "located_blocks.jsonl", located)
    write_jsonl(RUN_OUT / "unresolved_objects.jsonl", unresolved_images)
    write_jsonl(RUN_OUT / "page_audit.jsonl", page_audit)
    for name, chunks in experiments.items():
        assert len({c["chunk_id"] for c in chunks}) == len(chunks)
        assert all(c["text"] == corpus_text[c["start"]:c["end"]] for c in chunks)
        assert all(c["source_sha256"] == manifest["sha256"] for c in chunks)
        write_jsonl(RUN_OUT / f"chunks_{name}.jsonl", chunks)
    (RUN_OUT / "corpus.txt").write_text(corpus_text, encoding="utf-8")
    metrics.to_csv(RUN_OUT / "metrics.csv", index=False)
    evidence.to_csv(RUN_OUT / "evidence_preservation.csv", index=False)
    parsing_comparison.to_csv(RUN_OUT / "parsing_reference_comparison.csv", index=False)
    report = {**run_identity, "run_id": run_id, "created_at": datetime.now(UTC).isoformat(),
              "document_id": DOCUMENT_ID, "document_version": doc["version"],
              "blocks_total": len(blocks), "retrieval_blocks": len(located),
              "selected_characters": len(corpus_text), "parsing_seconds": parsing_seconds,
              "metrics": metric_rows, "encoder_executed": semantic_status["executed"],
              "llm_calls": 0, "review_status": reference["status"]}
    (RUN_OUT / "manifest.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n",
                                          encoding="utf-8")
    (OUT / "semantic_latest.json").write_text(json.dumps({"run_id": run_id, "path": str(RUN_OUT.relative_to(ROOT))},
                                              indent=2) + "\n", encoding="utf-8")
    print("Chunks reales y comparaciones exportados:", RUN_OUT)
    print("Índice vectorial creado: NO")

In [ ]:
fixed = metrics.set_index("experiment").loc["fixed_512_15"]
structure = metrics.set_index("experiment").loc["structural_512"]
display(Markdown(
    "## 10. Conclusiones calculadas de esta ejecución\n"
    f"- Corpus seleccionado: **{len(corpus_text):,} caracteres**, **{len(located)} bloques**.\n"
    f"- Fijo 512 con overlap 15%: **{int(fixed['chunks'])} chunks**, "
    f"factor de copias **{fixed['character_copy_factor']:.3f}**.\n"
    f"- Estructural 512: **{int(structure['chunks'])} chunks**, "
    f"**{int(structure['intact_table_groups'])}/{int(structure['table_groups'])}** grupos "
    f"de tabla íntegros; **{int(structure['chunks_over_encoder_window'])}** chunks superan 512.\n"
    f"- Experimento con encoder semántico ejecutado: **{semantic_status['executed']}**.\n"
    "- No se han medido mejoras de retrieval, fidelidad de respuestas ni exactitud clínica."
))

**Qué hemos aprendido:** contar con el tokenizer real y reservar prefijo/especiales;
cortar conservando offsets; medir redundancia; vincular tablas entre páginas y
distinguir un padre completo de un chunk que quepa en el encoder.

**Problemas abiertos:** mapa/jerarquía de secciones y transcripciones pendientes de
revisión humana; contenido de imágenes sin OCR; tablas sin reconstrucción semántica
de filas y notas; guiones de final de línea; muestra de referencia pequeña y sesgada.
Los resultados son mecánicos y específicos de cada documento por separado.

**Cuándo usar cada estrategia:** fijo como baseline controlable; overlap cuando se
acepte duplicación; estructural cuando límites documentales aportan unidades útiles,
manteniendo padres largos para diseñar hijos. La sección semántica es una hipótesis
ejecutable opcional, no una mejora demostrada.

**03:** instalar y ejecutar el encoder local, fijar la revisión de pesos, comparar
recuperación ES/EN sobre estos mismos chunks y anotar evidencia. Filtrar o tratar
explícitamente padres mayores de 512; nunca truncar la evidencia silenciosamente.
Aún no hace falta generar respuestas con un LLM para medir recuperación.

Ejercicios: inspeccionar el final de un chunk fijo que corte una tabla; variar overlap;
cambiar una región y observar cambios de IDs; diseñar hijos de una tabla con encabezado,
condición y nota preservados; revisar una ancla ausente en la imagen original.

Fuentes: [modelo y prefijos E5](https://huggingface.co/intfloat/multilingual-e5-base),
[Tokenizers](https://github.com/huggingface/tokenizers),
[pdfplumber](https://github.com/jsvine/pdfplumber). Las versiones ejecutadas se guardan
en el manifiesto. El cuaderno no descarga fuentes ni envía fragmentos a servicios Cloud.